# 📐 Evaluation as Specification
## Metrics ARE Software

*In traditional software, tests define correctness. In AI software, metrics define correctness.*

Your metric function IS your formal specification. It encodes what "good" means for your task — and the optimizer's job is to find prompts that satisfy that specification.

This notebook explores:
1. How metrics map to traditional test suites
2. Increasingly sophisticated metric functions
3. Cross-model evaluation as a diagnostic tool

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy
import ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.calculations import (METRIC_REGISTRY, code_execution_proxy, analogy_match,
                                      fact_verdict_accuracy, numeric_match)
from dspy_tasks.actions import run_baseline, _evaluate_examples, _mean
from dspy_tasks.visualize import *
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())
display(model_dd)

## Metrics Are Your Test Suite

| Traditional Software | AI Software |
|---------------------|-------------|
| Unit test | Metric function |
| `assert x == y` | `metric(expected, predicted) → score` |
| Pass / Fail | 0.0 to 1.0 (continuous quality) |
| Test suite | Dev set + metric |
| CI/CD gate | Evaluation threshold |

The shift is subtle but profound: in traditional software, tests are **binary** (pass/fail). In AI software, metrics are **continuous** — they measure *how good* the output is, not just whether it's correct.

This means your metric function encodes your **values**: what matters most, what trade-offs are acceptable, what "good enough" looks like.

In [ ]:
# Level 1: Binary exact match (simplest)
print("Level 1: Exact Match")
print(f"  'positive' == 'positive' → {1.0}")
print(f"  'positive' == 'POSITIVE' → {1.0}  (normalized)")
print(f"  'positive' == 'negative' → {0.0}")

# Level 2: Token F1 (partial credit)
from dspy_tasks.calculations import token_f1
print(f"\nLevel 2: Token F1")
print(f"  gold=['apple','banana'], pred=['apple','cherry'] → {token_f1(['apple','banana'], ['apple','cherry']):.3f}")

# Level 3: Composite weighted
print(f"\nLevel 3: Composite (ticket routing)")
print("  Priority correct (40%) + Category correct (35%) + Team correct (25%)")
print("  = Weighted specification of 'what matters most'")

display_insight("The Key Insight",
    "Your metric function IS your product specification. "
    "Changing the weights changes what the system optimizes for. "
    "This is why 'evaluation is the software of the future.'")

In [ ]:
from dspy_tasks.config import configure_dspy
# Code Generation task — different metric than simple matching
task = get_task("code_generation")
print(f"Task: {task.name}")
print(f"Metric: code_execution_proxy (checks structure, keywords, overlap)")
print(f"Teaching point: {task.teaching_point}\n")

btn = run_button("Evaluate Code Generation")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        configure_dspy(model=model_dd.value)
        result = run_baseline("code_generation", model_dd.value, max_eval=8)
        display_score("Code Generation", result.score)
        display_results_table(result.individual_scores)

btn.on_click(on_run)
display(widgets.HBox([model_dd, btn]), out)

In [ ]:
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task(tid) for tid in ["code_generation", "analogy", "fact_verification"]]],
    description="Task:")
compare_btn = run_button("Compare All Models")
compare_out = widgets.Output()

def on_compare(b):
    with compare_out:
        compare_out.clear_output()
        print(f"⏳ Evaluating {task_dd.value} across {len(MODELS)} models...")
        scores = {}
        for m in MODELS:
            result = run_baseline(task_dd.value, m, max_eval=8)
            scores[m] = {"baseline": result.score}
            display_score(m.split("/")[-1], result.score)

        fig = bar_comparison(get_task(task_dd.value).name, scores)
        fig.show()

compare_btn.on_click(on_compare)
display(widgets.HBox([task_dd, compare_btn]), compare_out)

## Different Metrics, Different Rankings

A critical insight: **the same model outputs can score differently under different metrics**.

- A model that produces verbose but accurate code might score high on `code_execution_proxy` but low on a conciseness metric.
- A model that gives terse answers might score high on exact match but low on token F1.

**Choosing your metric = choosing your values.** There is no "objectively best" model — only the best model *for your specification*.

In [ ]:
display_insight("Evaluation = Specification",
    "In traditional software, you write tests AFTER the code. "
    "In AI software, you write metrics BEFORE the optimization. "
    "The metric IS the specification. The optimizer finds code (prompts) that pass your tests.",
    icon="📐")